In [1]:
import os
from img_seg_v3 import *

In [19]:


def create_3d_blocks_large(data, block_size, filename, spath, overlap=1, datatype='uint8'):
    #print(block.shape)

    padding_needed = [0, 0]
    if data.shape[0] != block_size:
        padding_needed[0] = block_size - data.shape[0]
    if data.shape[1] != block_size:
        padding_needed[1] = block_size - data.shape[1]
    
    data = np.resize(data, (data.shape[0] + padding_needed[0], data.shape[1] + padding_needed[1], data.shape[2]))
    for l in range(0, data.shape[2], block_size - overlap):
        block = data[:, :, l: min(l + block_size, data.shape[2])]

        if block.shape[2] != block_size:
            block = np.resize(block, [block_size, block_size, block_size])

        
        fname = filename[:-4] + "_{:04d}".format(l) + '.tif'
        assert block.shape == (block_size, block_size, block_size)
        #if 1400 -  l < 64:
        #    print(block[30:32, 30:32, 1400 -  l:1400 -  l+2])
        #    break
            #print(block.shape, block[:2, :2, :2])

        save3dImages(block, spath, fname, datatype='uint8')

In [2]:
tpath = os.path.join(os.getcwd(), '..') 
frag_save_path = 'frag_results_3d'

seg_data_folder = os.path.join('CV', 'merged_cv_v2')

seg_flist,seg_fpath = readingfiles(os.path.join(tpath, 'results'),seg_data_folder)

seg_savepath = 'temp2'


In [49]:
avg_min_stack =0
avg_max_stack =0
number_stacks = len(org_flist)

for i in range(number_stacks):
    first_image = io.imread(os.path.join(org_fpath,org_flist[i]))
    original_size = first_image.shape
    g_min = np.min(first_image) 
    g_max = np.max(first_image)
    avg_min_stack += g_min / number_stacks
    avg_max_stack += g_max / number_stacks

g_avg_min = int(avg_min_stack)
g_avg_max = int(avg_max_stack)


In [5]:
org_flist,org_fpath = readingfiles(os.path.join(tpath,'sample_data'), 'original_images')
org_savepath = 'temp/'
block_size = 64

print(g_avg_min, g_avg_max)
n_files = len(org_flist)
# the last iteration only works with fewer slices and is resized in the create_3d_blocks function
for i in range(0, n_files, block_size - overlap):
    if i == 63:
        break
    rdata = []
    for j in range(0, original_size[0], block_size - overlap):
        rdata = [io.imread(os.path.join(org_fpath, org_flist[k]))[j: min(j + block_size, original_size[0])] for k in range(i, min(i + block_size, n_files))]
        rdata = np.array(rdata, dtype=float)

        np.clip(rdata, g_avg_min, g_avg_max, out=rdata)
        rdata -= g_avg_min
        rdata = ((255./(g_avg_max-g_avg_min)) * rdata).astype(np.uint8)

        f_name = org_flist[i][:-4] + "_{:04d}".format(j) + org_flist[i][-4:]
        if j == 0 or j == 63 * 6:
            print(j)
            print(os.path.join(org_fpath, org_flist[i]), rdata.shape)
            print(f_name)
            print(rdata[30:32, 30:32, 1400:1402])
            #print(rdata[:2, :2, :2])
            create_3d_blocks_large(rdata, block_size, filename=f_name, spath=org_savepath, overlap=overlap, datatype='uint8')

            if j == 63 * 6:
                break 
                
        #break

NameError: name 'g_avg_min' is not defined

In [21]:

n_files = len(seg_flist)
for i in range(0, n_files, block_size - overlap):
    if i == 63:
        break
    rdata = []
    for j in range(0, original_size[0], block_size - overlap):
        rdata = [io.imread(os.path.join(seg_fpath, seg_flist[k]))[j: min(j + block_size, original_size[0])] for k in range(i, min(i + block_size, n_files))]
        rdata = np.array(rdata, dtype=float)

        f_name = seg_flist[i][:-4] + "_{:04d}".format(j) + seg_flist[i][-4:]
        create_3d_blocks_large(rdata, block_size, filename=f_name, spath=seg_savepath, overlap=overlap, datatype='uint8')



In [4]:



#%% path setting for merged files
mfpath = os.path.join(tpath,'results','U_net',frag_save_path)
#mfpath = '../sample_data/U_net/training_data/seg_frag_data_3d'

mflist = os.listdir(mfpath)

merge_save_path = os.path.join(tpath, 'results', 'U_net', 'seg_merged_3d')
#merge_save_path = 'temp3_results'

filecounter = 0


#%%      
if not os.path.exists(merge_save_path):
    os.makedirs(merge_save_path)

#print(mflist[:100])

mflist.sort()

c_image_set = mflist[0].split('_')[3]
n_files = len(mflist)

block_size = 64
overlap = 1
original_size = [2940, 2940]
original_f_num = 2139

#print(reformed_image_set.shape, block_size * 2940 * 2940)

c = 0
print(n_files, c)
for i in range(0, original_f_num, block_size - overlap):
#while i < n_files:
    reformed_image_set = np.zeros([block_size] + original_size)

    if c >= n_files:
        break
    for j in range(0, original_size[0], block_size - overlap):
        if c >= n_files:
            break
        for k in range(0, original_size[1], block_size - overlap):
            if c >= n_files:
                break
            i_, j_, k_ = mflist[c].split('_')[3:]
            #print(i + 85 ,int(i_), j, int(j_), k, int(k_[:-4]))
            #assert i + 85 == int(i_) and j == int(j_) and k == int(k_[:-4])
            
            image = io.imread(os.path.join(mfpath, mflist[c]))
            space = reformed_image_set[i:i+64, j:j+64, k:k+64].shape
            if sum(space) != 64 * 3:
                
                print(i, j, k, space)
            for a in range(64):
                for b in range(64):
                    if a < space[0] and b < space[1]:
                        reformed_image_set[i+a][j+b][k:k+64] = image[a][b][:space[2]]
                        
            c += 1
            if c % 100 == 0:
                print(c / n_files)

                
    for l in range(block_size):
        i_, j_, k_ = mflist[c-1].split('_')[3:]

        if c-1 < n_files and l < len(reformed_image_set):
            f_name = os.path.join(merge_save_path, 'remerged_slice_{:04d}'.format(int(mflist[c-1].split('_')[3]) + l) + '.tif')
            io.imsave(f_name, reformed_image_set[l].astype('uint8'), check_contrast=False)


                
print('block done')

6627 0
(64, 64, 42)
(64, 64, 42)
0.01508978421608571
(64, 64, 42)
(64, 64, 42)
0.03017956843217142
(64, 64, 42)
(64, 64, 42)
0.04526935264825713
(64, 64, 42)
(64, 64, 42)
0.06035913686434284
(64, 64, 42)
(64, 64, 42)
0.07544892108042855
(64, 64, 42)
(64, 64, 42)
0.09053870529651425
(64, 64, 42)
(64, 64, 42)
0.10562848951259997
(64, 64, 42)
(64, 64, 42)
(64, 64, 42)
0.12071827372868568
(64, 64, 42)
(64, 64, 42)
0.13580805794477138
(64, 64, 42)
(64, 64, 42)
0.1508978421608571
(64, 64, 42)
(64, 64, 42)
0.1659876263769428
(64, 64, 42)
(64, 64, 42)
0.1810774105930285
(64, 64, 42)
(64, 64, 42)
0.19616719480911424
(64, 64, 42)
(64, 64, 42)
0.21125697902519994
(64, 64, 42)
(64, 64, 42)
0.22634676324128564
(64, 64, 42)
(64, 64, 42)
(64, 64, 42)
0.24143654745737136
(64, 64, 42)
(64, 64, 42)
0.25652633167345706
(64, 64, 42)
(64, 64, 42)
0.27161611588954276
(64, 64, 42)
(64, 64, 42)
0.28670590010562846
(64, 64, 42)
(64, 64, 42)
0.3017956843217142
(64, 64, 42)
(64, 64, 42)
0.3168854685377999
(64, 6

(1, 64, 42)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
0.4376037422664856
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 42)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 6

(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 42)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
0.5432322317790855
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 42)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 6

(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 42)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
0.6488607212916855
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 64)
(1, 64, 6

(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 42)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 42)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
0.76

(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 42)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
0.8752074845329711
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 42)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 64)
(0, 64, 6

(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
0.9959257582616569
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 64)
(0, 42, 42)
block done
